# Master Table
A table which holds all info about the games in one table

In [ ]:
import polars as pl
from pathlib import Path

# -----------------------------
# CONFIG
# -----------------------------
DATA_DIR = Path("../data/landing/march-machine-learning-mania-2025")

# -----------------------------
# LOAD CSVs
# -----------------------------
mteams = pl.read_csv(DATA_DIR / "MTeams.csv").select(["TeamID", "TeamName"])
wteams = pl.read_csv(DATA_DIR / "WTeams.csv").select(["TeamID", "TeamName"])
teams = pl.concat([mteams, wteams])

mconfs = pl.read_csv(DATA_DIR / "MTeamConferences.csv")
wconfs = pl.read_csv(DATA_DIR / "WTeamConferences.csv")
confs = pl.concat([mconfs, wconfs])

mcoaches = pl.read_csv(DATA_DIR / "MTeamCoaches.csv")

mseeds = pl.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
wseeds = pl.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

mmassey = pl.read_csv(DATA_DIR / "MMasseyOrdinals.csv")

m_reg = pl.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
w_reg = pl.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
m_tour = pl.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")
w_tour = pl.read_csv(DATA_DIR / "WNCAATourneyDetailedResults.csv")

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def clean_seeds(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.col("Seed").str.slice(1, 2).cast(pl.Int32).alias("SeedNum")
    )

def add_team_names(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.join(teams.rename({"TeamID": "WTeamID", "TeamName": "WTeamName"}), on="WTeamID", how="left")
          .join(teams.rename({"TeamID": "LTeamID", "TeamName": "LTeamName"}), on="LTeamID", how="left")
    )

def add_conferences(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.join(confs.rename({"TeamID": "WTeamID", "ConfAbbrev": "WConference"}), on=["Season", "WTeamID"], how="left")
          .join(confs.rename({"TeamID": "LTeamID", "ConfAbbrev": "LConference"}), on=["Season", "LTeamID"], how="left")
    )

def add_seeds(df: pl.DataFrame) -> pl.DataFrame:
    seeds = pl.concat([mseeds, wseeds])

    # Winning team seeds
    df = df.join(
        seeds.rename({"TeamID": "WTeamID", "SeedNum": "WSeed"}),
        on=["Season", "WTeamID"],
        how="left"
    ).with_columns([
        pl.when(pl.col("IsTournament") == False).then(None).otherwise(pl.col("WSeed")).alias("WSeed")
    ])

    # Losing team seeds
    df = df.join(
        seeds.rename({"TeamID": "LTeamID", "SeedNum": "LSeed"}),
        on=["Season", "LTeamID"],
        how="left"
    ).with_columns([
        pl.when(pl.col("IsTournament") == False).then(None).otherwise(pl.col("LSeed")).alias("LSeed")
    ])

    return df


def add_coaches_men_only(df: pl.DataFrame) -> pl.DataFrame:
    df = (
        df.join(mcoaches.rename({"TeamID": "WTeamID", "CoachName": "WCoach"}), on=["Season", "WTeamID"], how="left")
          .join(mcoaches.rename({"TeamID": "LTeamID", "CoachName": "LCoach"}), on=["Season", "LTeamID"], how="left")
    )
    return df.with_columns([
        pl.when(pl.col("Gender") == "W").then(None).otherwise(pl.col("WCoach")).alias("WCoach"),
        pl.when(pl.col("Gender") == "W").then(None).otherwise(pl.col("LCoach")).alias("LCoach"),
    ])

def add_massey_men_only(df: pl.DataFrame) -> pl.DataFrame:
    """
    Adds WMassey/LMassey from POM rankings as-of each game DayNum.
    If the ranking is not yet released for the DayNum, put None.
    Women's games get None.
    """
    # Filter men's Massey POM rankings
    massey = (
        mmassey.filter(pl.col("SystemName") == "POM")
                .select(["Season", "TeamID", "RankingDayNum", "OrdinalRank"])
                .sort(["Season", "TeamID", "RankingDayNum"])
    )

    # -----------------------------
    # Winning team
    # -----------------------------
    w = df.select(["Season", "WTeamID", "DayNum"])
    w_massey = w.join(
        massey.rename({"TeamID": "WTeamID", "OrdinalRank": "WMassey"}),
        on="WTeamID",
        how="left"
    ).with_columns([
        # Keep ranking only if released, else None
        pl.when(pl.col("RankingDayNum") <= pl.col("DayNum"))
          .then(pl.col("WMassey"))
          .otherwise(None)
          .cast(pl.Int32)   # ensure nullable integer
          .alias("WMassey")
    ])

    w_massey = (
        w_massey.sort([
            pl.col("Season"),
            pl.col("WTeamID"),
            pl.col("DayNum"),
            pl.col("RankingDayNum").reverse()
        ])
        .unique(subset=["Season", "WTeamID", "DayNum"])
        .select(["Season", "WTeamID", "DayNum", "WMassey"])
    )

    # -----------------------------
    # Losing team
    # -----------------------------
    l = df.select(["Season", "LTeamID", "DayNum"])
    l_massey = l.join(
        massey.rename({"TeamID": "LTeamID", "OrdinalRank": "LMassey"}),
        on="LTeamID",
        how="left"
    ).with_columns([
        pl.when(pl.col("RankingDayNum") <= pl.col("DayNum"))
          .then(pl.col("LMassey"))
          .otherwise(None)
          .cast(pl.Int32)
          .alias("LMassey")
    ])

    l_massey = (
        l_massey.sort([
            pl.col("Season"),
            pl.col("LTeamID"),
            pl.col("DayNum"),
            pl.col("RankingDayNum").reverse()
        ])
        .unique(subset=["Season", "LTeamID", "DayNum"])
        .select(["Season", "LTeamID", "DayNum", "LMassey"])
    )

    # -----------------------------
    # Join back to main df
    # -----------------------------
    df = df.join(w_massey, on=["Season", "WTeamID", "DayNum"], how="left")
    df = df.join(l_massey, on=["Season", "LTeamID", "DayNum"], how="left")

    # Nullify women's games
    df = df.with_columns([
        pl.when(pl.col("Gender") == "W").then(None).otherwise(pl.col("WMassey")).alias("WMassey"),
        pl.when(pl.col("Gender") == "W").then(None).otherwise(pl.col("LMassey")).alias("LMassey"),
    ])

    return df



def prep_results(df, gender, is_tourney):
    return df.with_columns([
        pl.lit(gender).alias("Gender"),
        pl.lit(is_tourney).alias("IsTournament")
    ])

# -----------------------------
# CLEAN & PREP
# -----------------------------
mseeds = clean_seeds(mseeds)
wseeds = clean_seeds(wseeds)

m_reg = prep_results(m_reg, "M", False)
w_reg = prep_results(w_reg, "W", False)
m_tour = prep_results(m_tour, "M", True)
w_tour = prep_results(w_tour, "W", True)

all_results = pl.concat([m_reg, w_reg, m_tour, w_tour], how="diagonal")

# -----------------------------
# ADD METADATA
# -----------------------------
all_results = add_team_names(all_results)
all_results = add_conferences(all_results)
all_results = add_seeds(all_results)
all_results = add_coaches_men_only(all_results)
all_results = add_massey_men_only(all_results)

# -----------------------------
# REORDER COLUMNS
# -----------------------------
priority_cols = [
    "Season", "Gender", "IsTournament", "DayNum",
    "WTeamName", "LTeamName",
    "WScore", "LScore",
    "WSeed", "LSeed",
    "WConference", "LConference",
    "WCoach", "LCoach",
    "WMassey", "LMassey",
    "WLoc", "NumOT"
]
remaining = [c for c in all_results.columns if c not in priority_cols]
final_table = all_results.select(priority_cols + remaining)

display(final_table)


Season,Gender,IsTournament,DayNum,WTeamName,LTeamName,WScore,LScore,WSeed,LSeed,WConference,LConference,WCoach,LCoach,WMassey,LMassey,WLoc,NumOT,WTeamID,LTeamID,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,Seed,Seed_right,FirstDayNum,LastDayNum,FirstDayNum_right,LastDayNum_right
i64,str,bool,i64,str,str,i64,i64,i32,i32,str,str,str,str,i32,i32,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,i64,i64,i64,i64
2003,"""M""",false,10,"""Alabama""","""Oklahoma""",68,62,10,1,"""sec""","""big_twelve""","""mark_gottfried""","""kelvin_sampson""",null,null,"""N""",0,1104,1328,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20,"""Y10""","""W01""",0,154,0,154
2003,"""M""",false,10,"""Memphis""","""Syracuse""",70,63,7,3,"""cusa""","""big_east""","""john_calipari""","""jim_boeheim""",null,null,"""N""",0,1272,1393,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16,"""Z07""","""W03""",0,154,0,154
2003,"""M""",false,11,"""Marquette""","""Villanova""",73,61,3,null,"""cusa""","""big_east""","""tom_crean""","""jay_wright""",null,null,"""N""",0,1266,1437,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23,"""Y03""",null,0,154,0,154
2003,"""M""",false,11,"""N Illinois""","""Winthrop""",56,50,null,null,"""mac""","""big_south""","""rob_judson""","""gregg_marshall""",null,null,"""N""",0,1296,1457,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23,null,null,0,154,0,154
2003,"""M""",false,11,"""Texas""","""Georgia""",77,71,1,null,"""big_twelve""","""sec""","""rick_barnes""","""jim_harrick""",null,null,"""N""",0,1400,1208,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14,"""X01""",null,0,154,0,154
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024,"""W""",true,147,"""Connecticut""","""USC""",80,73,3,1,"""big_east""","""pac_twelve""",null,null,null,null,"""A""",0,3163,3425,28,58,7,15,17,27,5,30,17,12,6,5,21,23,70,9,29,18,20,10,25,10,9,6,4,20,"""Z03""","""Z01""",null,null,null,null
2024,"""W""",true,147,"""Iowa""","""LSU""",94,87,1,3,"""big_ten""","""sec""",null,null,null,null,"""H""",0,3234,3261,32,69,13,31,17,22,3,29,16,11,6,3,15,34,88,8,24,11,17,21,28,15,13,6,6,21,"""Y01""","""Y03""",null,null,null,null
2024,"""W""",true,151,"""Iowa""","""Connecticut""",71,69,1,3,"""big_ten""","""big_east""",null,null,null,null,"""N""",0,3234,3163,27,59,7,25,10,14,9,23,12,16,7,1,9,29,63,8,25,3,4,6,22,21,14,15,1,18,"""Y01""","""Z03""",null,null,null,null
